# Petrol Stations in Switzerland – Competitive Analysis

This Jupyter notebook does the following:

- Retrieve petrol station locations in the Canton of Zurich from OpenStreetMap using the Overpass API, with retries and a fallback endpoint.
- Extract station names, operators, brands, and coordinates into a pandas DataFrame.
- Save the raw response as JSON and the station table as CSV.
- Count stations by operator and list the five largest operators in the retrieved data.
- Display an interactive station map and save it as HTML.

## Libraries and settings

In [1]:
# Libraries
import os
import json
import requests
import folium
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

/workspaces/spatial_data_analysis/01_Python_Basic_GIS_Functionalities


## Fetch petrol station data from OpenStreetMap (Overpass API)

Note: If a JSONDecodeError occurs, try again this is most probably because the Overpass server is overloaded

In [2]:
# Overpass API endpoints (main server + Swiss mirror as fallback in case of overload)
overpass_urls = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.osm.ch/api/interpreter',
]

# Query: fuel nodes within canton Zurich (smaller area for faster response)
query = """
[out:json];
area["ISO3166-2"="CH-ZH"][admin_level=4];
node["amenity"="fuel"](area);
out;
"""

# Send request with retry logic, trying each endpoint in turn
import time

max_retries = 5
data = None
for overpass_url in overpass_urls:
    print(f'Trying Overpass endpoint: {overpass_url}')
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(overpass_url, params={'data': query}, timeout=60)
            data = response.json()
            print(f"Number of petrol stations retrieved: {len(data['elements'])}")
            break
        except Exception:
            if attempt < max_retries:
                print("Server may be overloaded — retrying in 10 seconds...")
                time.sleep(10)
            else:
                print(f'Giving up on {overpass_url} after {max_retries} attempts.')
    if data is not None:
        break

if data is None:
    raise RuntimeError("Max retries reached on all Overpass endpoints. Please try again later.")

Trying Overpass endpoint: https://overpass-api.de/api/interpreter
Server may be overloaded — retrying in 10 seconds...
Server may be overloaded — retrying in 10 seconds...
Server may be overloaded — retrying in 10 seconds...
Server may be overloaded — retrying in 10 seconds...
Giving up on https://overpass-api.de/api/interpreter after 5 attempts.
Trying Overpass endpoint: https://overpass.osm.ch/api/interpreter
Number of petrol stations retrieved: 307


## Save raw data as JSON

In [3]:
# Save JSON
json_path = 'petrol_stations_ch.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f'JSON saved to: {json_path}')

JSON saved to: petrol_stations_ch.json


## Parse JSON and create a DataFrame

In [4]:
# Extract relevant fields from each node
records = []
for element in data['elements']:
    tags = element.get('tags', {})
    records.append({
        'id':       element.get('id'),
        'lat':      element.get('lat'),
        'lon':      element.get('lon'),
        'name':     tags.get('name', 'unknown'),
        'operator': tags.get('operator', 'unknown'),
        'brand':    tags.get('brand', 'unknown'),
    })

df = pd.DataFrame(records)
print(f'Shape: {df.shape}')
df.head()

Shape: (307, 6)


,id,lat,lon,name,operator,brand
0,27069561,47.396391,8.623317,SOCAR Zur Schmiede,SOCAR Energy Switzerland,SOCAR
1,27125026,47.401093,8.619955,BP Garage Tresch,Tresch Automobile,BP
2,34078845,47.495696,8.712981,Migrol Service Zürcherstrasse,Migrol,Shell
3,47744371,47.399526,8.497329,Coop Tankstelle Höngg,Coop Mineraloel,Coop
4,52741787,47.381593,8.509545,Shell Hardau,Shell (Switzerland),Shell


## Save data as CSV

In [5]:
# Save CSV
csv_path = 'petrol_stations_ch.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')

print(f'CSV saved to: {csv_path}')

CSV saved to: petrol_stations_ch.csv


## Count petrol stations per operator (pivot table)

In [6]:
# Pivot table: count stations per operator, sorted descending
pivot = df.pivot_table(
    index='operator',
    values='id',
    aggfunc='count'
).rename(columns={'id': 'count'}).sort_values('count', ascending=False)

print(f'Total operators: {len(pivot)}')
print(pivot.head(10))

Total operators: 63
                          count
operator                       
unknown                     113
Migrol                       32
Avia Osterwalder Zürich      21
SOCAR Energy Switzerland     18
Coop Mineraloel              15
A.H. Meyer & Cie             12
Shell (Switzerland)          10
Agrola                        6
Schweizer Armee               5
Moveri                        5


## Top 5 largest operators

In [7]:
# Top 5 operators
top5 = pivot.head(5).reset_index()
top5.columns = ['Operator', 'Number of Stations']
top5.index = top5.index + 1  # start ranking at 1

print('Top 5 petrol station operators in Switzerland:')
top5

Top 5 petrol station operators in Switzerland:


,Operator,Number of Stations
1,unknown,113
2,Migrol,32
3,Avia Osterwalder Zürich,21
4,SOCAR Energy Switzerland,18
5,Coop Mineraloel,15


## Visualise all petrol stations on an interactive map

In [8]:
# Initialise map centred on canton Zurich
m = folium.Map(location=[47.41, 8.65], zoom_start=10, tiles='OpenStreetMap')

# Add a marker for each petrol station
for _, row in df.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        icon=folium.Icon(color='green', icon='tint', prefix='fa'),
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>Operator: {row['operator']}<br>Brand: {row['brand']}",
            max_width=200
        )
    ).add_to(m)

# Save map as HTML
m.save('petrol_stations_ch_map.html')

# Display map
m

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [9]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1070-azure
Datetime: 2026-11-27 09:27:36
Python Version: 3.12.16
-----------------------------------
